# Stage 1b — paired KV cross-subspaces

Stage 1 found stable residual directions that were equally stable after shuffling. This notebook removes those marginal teacher and student covariances by measuring whitened cross-covariance directly. It compares correctly paired teacher/student trajectories with repeated within-batch teacher derangements.

No weights are updated. Extraction is restartable and all durable artifacts are written to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/0x0shephard/latent-reasoning.git'
RUN_COMMIT = 'main'  # Replace with the printed SHA before the long extraction.
REPO_DIR = '/content/latent-reasoning'
DRIVE_ROOT = '/content/drive/MyDrive/CODI_KAVA'
KAVA_OUTPUT = f'{DRIVE_ROOT}/outputs/kava'
STAGE1B_OUTPUT = f'{DRIVE_ROOT}/outputs/stage1b_kv_cross_subspaces'
REPORT_JSON = f'{DRIVE_ROOT}/reports/stage1b_kv_cross_subspaces.json'

CALIBRATION_EXAMPLES = 2000
BATCH_SIZE = 4
NUM_SPLITS = 2
SHUFFLE_REPEATS = 4
SAVE_EVERY = 500
SEED = 0
RUN_5000_CONFIRMATION = False

In [ ]:
import os, pathlib, subprocess, sys

os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '300'
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
        print('Hugging Face authentication: Colab secret loaded')
except Exception:
    print('Hugging Face authentication: public access')

if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
target = 'origin/main' if RUN_COMMIT == 'main' else RUN_COMMIT
subprocess.run(['git', '-C', REPO_DIR, 'checkout', '--detach', target], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-r', os.path.join(REPO_DIR, 'requirements.txt')
], check=True)
os.chdir(REPO_DIR)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Checked out:', commit)
if RUN_COMMIT == 'main':
    print('Pin RUN_COMMIT to this SHA:', commit)
for required in ('scripts/collect_kv_cross_subspaces.py', 'scripts/analyze_kv_cross_subspaces.py'):
    assert pathlib.Path(required).is_file(), f'Missing {required}; push the Stage 1b changes first'

In [ ]:
import json, torch
from pathlib import Path

assert torch.cuda.is_available(), 'Enable a Colab GPU runtime'
print('Torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
kava_root = Path(KAVA_OUTPUT)
manifest_path = kava_root / 'run_manifest.json'
checkpoints = sorted((kava_root / 'checkpoints').glob('step_*.pt'))
assert manifest_path.is_file()
assert checkpoints
latest = checkpoints[-1]
step = int(latest.stem.split('_')[1])
manifest = json.loads(manifest_path.read_text())
task = manifest['effective_config']['task']
assert task['method'] == 'kava'
assert float(task['distillation']['kv_weight']) > 0
assert step == 96405, f'Expected step 96405, found {step}'
print('Verified checkpoint:', latest)
print('Fingerprint:', manifest['fingerprint'])

In [ ]:
import datetime, time

logs = pathlib.Path(DRIVE_ROOT) / 'logs' / 'stage1b_kv_cross_subspaces'
logs.mkdir(parents=True, exist_ok=True)

def run_logged(cmd, log_name):
    log_path = logs / log_name
    print('Starting:', ' '.join(map(str, cmd)), flush=True)
    print('Persistent log:', log_path, flush=True)
    with log_path.open('a', encoding='utf-8', buffering=1) as log:
        log.write(f"\n=== {datetime.datetime.now(datetime.timezone.utc).isoformat()} {' '.join(map(str, cmd))} ===\n")
        process = subprocess.Popen(
            list(map(str, cmd)), cwd=REPO_DIR,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        last_flush = time.monotonic()
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            if time.monotonic() - last_flush >= 30:
                log.flush()
                last_flush = time.monotonic()
        code = process.wait()
        log.flush()
    if code != 0:
        raise RuntimeError(f'Command failed with exit code {code}. See {log_path}')
    return code

In [ ]:
collect_cmd = [
    sys.executable, '-u', 'scripts/collect_kv_cross_subspaces.py',
    '--config', 'configs/kava.yaml',
    '--checkpoint-root', KAVA_OUTPUT,
    '--output-dir', STAGE1B_OUTPUT,
    '--examples', str(CALIBRATION_EXAMPLES),
    '--batch-size', str(BATCH_SIZE),
    '--num-splits', str(NUM_SPLITS),
    '--shuffle-repeats', str(SHUFFLE_REPEATS),
    '--save-every', str(SAVE_EVERY),
    '--seed', str(SEED),
    '--precision', 'auto',
]
run_logged(collect_cmd, f'collect_n{CALIBRATION_EXAMPLES}.log')

In [ ]:
analyze_cmd = [
    sys.executable, 'scripts/analyze_kv_cross_subspaces.py',
    '--statistics', STAGE1B_OUTPUT,
    '--output', REPORT_JSON,
]
run_logged(analyze_cmd, f'analyze_n{CALIBRATION_EXAMPLES}.log')

from IPython.display import Markdown, display
report_md = pathlib.Path(REPORT_JSON).with_suffix('.md')
display(Markdown(report_md.read_text(encoding='utf-8')))

In [ ]:
# Optional confirmation. It extends the same deterministic sample prefix.
if RUN_5000_CONFIRMATION:
    confirmation_cmd = collect_cmd.copy()
    confirmation_cmd[confirmation_cmd.index('--examples') + 1] = '5000'
    run_logged(confirmation_cmd, 'collect_n5000.log')
    run_logged(analyze_cmd, 'analyze_n5000.log')
    display(Markdown(report_md.read_text(encoding='utf-8')))
else:
    print('5,000-example confirmation skipped.')

In [ ]:
print('\nDurable Stage 1b artifacts')
for root in (pathlib.Path(STAGE1B_OUTPUT), pathlib.Path(REPORT_JSON).parent):
    print(f'\n{root}')
    paths = sorted(root.rglob('*')) if root.name != 'reports' else sorted(root.glob('stage1b_kv_cross_subspaces*'))
    for path in paths:
        if path.is_file():
            print(f'  {path.relative_to(root)}  {path.stat().st_size / (1024**2):.1f} MiB')